# 9 · Working with Dates & Times
*Intro to Python for Scientists & Public Health Professionals*

Time is central to public-health data — cases per week, visits per month, trends over years. pandas has rich datetime tools: parse strings to real dates, pull out parts with the `.dt` accessor, and — most powerfully — **resample** a time series to any frequency.

We'll use a year of daily clinic-visit counts for three sites.

### By the end of this notebook you can
- Parse strings to datetimes, including custom formats and bad values
- Extract date parts with `.dt` (year, month, day-of-week, week-of-year)
- Use a datetime index to slice by date and resample to weekly/monthly
- Compute rolling averages and handle gaps in a time series

### Agenda
1. Parsing dates
2. The `.dt` accessor
3. Datetime as the index
4. Resampling
5. Rolling windows & day-of-week patterns
6. Missing values in a time series

### How the exercises work
Each exercise has a prompt, an empty cell to try it yourself, and a collapsed **Solution** you can expand to check your work.

In [ ]:
import numpy as np
import pandas as pd

BASE_URL = "https://raw.githubusercontent.com/jimcody2014/2026-python-data/refs/heads/main"

## 1. Parsing dates

`pd.to_datetime` converts strings to real datetimes. It infers common formats, but you can be explicit, and coerce unparseable values to `NaT` (the datetime version of `NaN`).

In [ ]:
# pd.to_datetime("2025-03-10")                       # ISO format, inferred
# pd.to_datetime("10/03/2025", dayfirst=True)        # day-first (10 March)
# pd.to_datetime("2025-31-12", format="%Y-%d-%m")    # explicit format

In [ ]:
# errors="coerce" turns anything unparseable into NaT instead of raising
pd.to_datetime(["2025-03-10", "not a date", "2025-03-12"], errors="coerce")

Most often you parse a whole column at read time with `parse_dates`.

In [ ]:
visits = pd.read_csv(f"{BASE_URL}/clinic_visits.csv", parse_dates=["date"])
print(visits.dtypes)
visits.head()

> Reference: [`to_datetime`](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html).

## 2. The `.dt` accessor

Once a column is datetime, `.dt` exposes its parts.

In [ ]:
visits["year"]  = visits["date"].dt.year
visits["month"] = visits["date"].dt.month
visits["dow"]   = visits["date"].dt.day_name()
visits["week"]  = visits["date"].dt.isocalendar().week
visits[["date", "year", "month", "dow", "week"]].head()

## 3. Datetime as the index

Putting the date on the index unlocks date-string slicing and resampling. Our data is *long* (three sites share each date), which is fine — operations aggregate across the matching rows.

In [ ]:
ts = visits.set_index("date").sort_index()
ts.loc["2025-03"].head()        # every row in March (all three sites)

In [ ]:
ts.loc["2025-03":"2025-06"].shape   # a date range — inclusive of both ends

## 4. Resampling

`resample` is `groupby` for time: it buckets rows into calendar periods, then aggregates. `"ME"` = month-end, `"W"` = weekly, `"D"` = daily.

In [ ]:
# total visits per month, summed across all sites
ts["visits"].resample("ME").sum()

In [ ]:
# per site, per month — combine groupby with resample
ts.groupby("site").resample("ME")["visits"].sum().head(9)

> Reference: [Resampling](https://pandas.pydata.org/docs/user_guide/timeseries.html#resampling) and [frequency aliases](https://pandas.pydata.org/docs/user_guide/timeseries.html#offset-aliases).

## 5. Rolling windows & day-of-week patterns

A rolling mean smooths daily noise into a trend. Here's a 7-day average for one site.

In [ ]:
riverside = ts[ts["site"] == "Riverside"].sort_index()
riverside["visits_7day_avg"] = riverside["visits"].rolling(7).mean()
riverside[["visits", "visits_7day_avg"]].head(9)

Grouping by the day name reveals the weekly rhythm — busy weekdays, light Saturdays, closed Sundays.

In [ ]:
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
riverside.groupby(riverside.index.day_name())["visits"].mean().reindex(order).round(0)

## 6. Missing values in a time series

Two kinds of gaps live in this data: Sundays are closed (so wait times are `NaN`), and one site had a 5-day logging outage in March.

In [ ]:
hilltop = ts[ts["site"] == "Hilltop"].sort_index()
hilltop.loc["2025-03-08":"2025-03-16", ["visits", "avg_wait_min"]]

For an ordered series, `interpolate` estimates the gap from the values on either side — often more sensible than a flat fill for time data.

In [ ]:
hilltop["visits_filled"] = hilltop["visits"].interpolate()
hilltop.loc["2025-03-08":"2025-03-16", ["visits", "visits_filled"]]

> Other options: `ffill()` / `bfill()` carry the last/next known value across the gap.

### Exercise 1 — Weekly totals *(6 min)*

Using `ts`, compute the **total visits per week** across all sites.

In [ ]:
# Your work here


### Exercise 2 — Day-of-week pattern *(6 min)*

For the **Downtown** site, compute the mean visits for each day of the week, in calendar order.

In [ ]:
# Your work here


## Wrap-up

You can parse dates, extract their parts with `.dt`, use a datetime index to slice and resample, smooth with rolling windows, and fill gaps in a time series.

**Next:** Combining datasets — merge, join, and concatenate.